# Spike Hunter V3: Advanced Binance-Style AI Insights Pipeline
A full end-to-end notebook implementing real-time spike detection, adaptive thresholds, ML modeling, explanation, and narrative insight scaffolding.

Outline checkpoints implemented sequentially below.

## 1. Environment Setup and Dependency Installation
Installs and verifies required libraries. Avoids hard fail if already installed.
Dependencies: python-binance, websockets, pandas, numpy, lightgbm, xgboost, scikit-learn, plotly, shap, pyarrow.
A GPU check is optional.


In [ ]:
import importlib, sys, subprocess, json, os, platform, time
REQUIRED_LIBS = [
    "python-binance", "websockets", "pandas", "numpy", "lightgbm", "xgboost", "scikit-learn", "plotly", "shap", "pyarrow"
]

def ensure_libs(libs):
    for lib in libs:
        pkg = lib.split("[")[0]
        try:
            importlib.import_module(pkg.replace('-', '_'))
        except ImportError:
            print(f"[Install] {lib}")
            subprocess.run([sys.executable, "-m", "pip", "install", lib], check=False)

ensure_libs(REQUIRED_LIBS)

versions = {}
for lib in ["pandas", "numpy", "lightgbm", "xgboost", "sklearn", "plotly", "shap"]:
    try:
        module = importlib.import_module(lib if lib != "sklearn" else "sklearn")
        versions[lib] = getattr(module, "__version__", "unknown")
    except Exception:
        versions[lib] = None

print("[Versions]", json.dumps(versions, indent=2))
print("[Python]", sys.version)
print("[Platform]", platform.platform())

# Optional GPU check
try:
    import lightgbm as lgb
    gpu_available = any("gpu" in d.lower() for d in lgb.get_device_name().split()) if hasattr(lgb, 'get_device_name') else False
    print("[LightGBM GPU]", gpu_available)
except Exception:
    print("[LightGBM GPU] check failed")

## 2. Configuration and Secure API Key Handling
Load API keys via environment variables (BINANCE_API_KEY, BINANCE_API_SECRET). Optional .env fallback. Never hardcode keys.

In [ ]:
from pathlib import Path
from typing import Optional

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

API_KEY = os.getenv("BINANCE_API_KEY")
API_SECRET = os.getenv("BINANCE_API_SECRET")

def assert_keys():
    if not API_KEY or not API_SECRET:
        raise EnvironmentError("Missing BINANCE_API_KEY or BINANCE_API_SECRET env vars.")
    print("[Keys] Loaded API key & secret from environment.")

# Lazy assertion (only if private endpoints / signed streams used later)
print("[Config] API key present?", bool(API_KEY))


## 3. Binance REST Data Fetch (Historical OHLCV)
Minimal fetch function using python-binance client or raw HTTP fallback.

In [ ]:
import requests
import pandas as pd
import numpy as np

BINANCE_BASE = "https://api.binance.com"

def fetch_klines(symbol: str, interval: str = "1m", limit: int = 500) -> pd.DataFrame:
    url = f"{BINANCE_BASE}/api/v3/klines"
    params = {"symbol": symbol.upper(), "interval": interval, "limit": limit}
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    cols = [
        "open_time", "open", "high", "low", "close", "volume", "close_time", "quote_asset_volume",
        "number_of_trades", "taker_buy_base", "taker_buy_quote", "ignore"
    ]
    df = pd.DataFrame(data, columns=cols)
    for c in ["open", "high", "low", "close", "volume", "quote_asset_volume", "taker_buy_base", "taker_buy_quote"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["timestamp"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)
    df = df[["timestamp", "open", "high", "low", "close", "volume", "quote_asset_volume", "number_of_trades", "taker_buy_base", "taker_buy_quote"]]
    return df.dropna().reset_index(drop=True)

seed_df = fetch_klines("BTCUSDT", interval="1m", limit=300)
print("[Seed] Fetched rows:", len(seed_df))
seed_df.head()

## 4. WebSocket Live Stream (Trades + Klines)
Async streaming of trade & kline updates with reconnect and backoff.

In [ ]:
import asyncio, json, websockets, random
from collections import deque

STREAM_BASE = "wss://stream.binance.com:9443/ws"

class LiveStream:
    def __init__(self, symbol: str, interval: str = "1m", max_queue: int = 2000):
        self.symbol = symbol.lower()
        self.interval = interval
        self.trade_queue = deque(maxlen=max_queue)
        self.kline_queue = deque(maxlen=max_queue)
        self._stop = False

    async def _connect(self, path: str):
        backoff = 1
        while not self._stop:
            try:
                async with websockets.connect(path, ping_interval=15, ping_timeout=10) as ws:
                    print(f"[Stream] Connected {path}")
                    backoff = 1
                    async for msg in ws:
                        data = json.loads(msg)
                        self._handle_message(data)
                        if self._stop:
                            break
            except Exception as e:
                print(f"[Stream] Error {e}; reconnect in {backoff}s")
                await asyncio.sleep(backoff)
                backoff = min(backoff * 2, 60)

    def _handle_message(self, data: dict):
        if "e" in data and data.get("e") == "trade":
            self.trade_queue.append(data)
        elif "k" in data:  # kline stream wrapper
            self.kline_queue.append(data["k"])

    async def run(self):
        trade_stream = f"{self.symbol}@trade"
        kline_stream = f"{self.symbol}@kline_{self.interval}"
        path = f"{STREAM_BASE}/{trade_stream}"  # Basic single stream; could multiplex
        task_trade = asyncio.create_task(self._connect(path))
        # For simplicity using separate connection for kline
        path_k = f"{STREAM_BASE}/{kline_stream}"
        task_k = asyncio.create_task(self._connect(path_k))
        await asyncio.wait([task_trade, task_k])

    def stop(self):
        self._stop = True

# NOTE: Not starting event loop automatically in notebook to avoid blocking.
print("[Stream] LiveStream class ready (manual asyncio.run() to activate).")

## 5. Data Normalization and Time Alignment
Merge historical seed with streaming updates; assemble uniform interval bars and forward-fill partial data.

In [ ]:
def normalize_bars(df: pd.DataFrame, interval: str = "1min") -> pd.DataFrame:
    work = df.copy().set_index("timestamp").sort_index()
    ohlc = work[["open", "high", "low", "close", "volume"]]
    agg = ohlc.resample(interval).agg({
        "open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"
    })
    agg.ffill(inplace=True)
    if "quote_asset_volume" in work.columns:
        qv = work["quote_asset_volume"].resample(interval).sum()
        agg["quote_asset_volume"] = qv
    if "number_of_trades" in work.columns:
        trades = work["number_of_trades"].resample(interval).sum()
        agg["number_of_trades"] = trades
    agg = agg.dropna().reset_index()
    return agg

normalized_df = normalize_bars(seed_df, interval="1min")
print("[Normalize] Rows ->", len(seed_df), "to", len(normalized_df))
normalized_df.head()

## 6. Return, Volatility, and Liquidity Feature Engineering
Compute log returns, rolling realized volatility, volume deltas, wick ratios, simple spread proxy.

In [ ]:
def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy().sort_values("timestamp")
    work["log_ret"] = np.log(work["close"].pct_change() + 1).replace([np.inf, -np.inf], np.nan)
    work["raw_ret"] = work["close"].pct_change()
    # Rolling realized volatility (window=20)
    work["realized_vol_20"] = work["raw_ret"].rolling(20).std() * np.sqrt(20)
    # EWMA volatility
    alpha = 0.2
    ewma = []
    prev = 0.0
    for r in work["raw_ret"].fillna(0).abs():
        val = alpha * r + (1 - alpha) * prev
        ewma.append(val)
        prev = val
    work["ewma_vol"] = ewma
    # Wick ratios
    body = (work["close"] - work["open"]).abs()
    upper_wick = work["high"] - work[["close", "open"]].max(axis=1)
    lower_wick = work[["close", "open"]].min(axis=1) - work["low"]
    work["upper_wick_ratio"] = upper_wick / (body + 1e-6)
    work["lower_wick_ratio"] = lower_wick / (body + 1e-6)
    # Volume delta
    work["volume_delta"] = work["volume"].diff()
    work["volume_ma_20"] = work["volume"].rolling(20).mean()
    work["volume_ratio_20"] = work["volume"] / (work["volume_ma_20"] + 1e-6)
    # Spread proxy (high-low normalized)
    work["spread_pct"] = (work["high"] - work["low"]) / (work["close"] + 1e-6)
    return work

features_df = add_base_features(normalized_df)
print("[Features] Columns added:", set(features_df.columns) - set(normalized_df.columns))
features_df.tail(3)

## 7. Spike Scoring Function (Z-Score, EWMA Volatility)
Compute spike_score = |return| / ewma_vol and z-score variant.

In [ ]:
def compute_spike_scores(df: pd.DataFrame, ret_col: str = "raw_ret", ewma_col: str = "ewma_vol") -> pd.DataFrame:
    work = df.copy()
    work["spike_score"] = (work[ret_col].abs() / (work[ewma_col] + 1e-6)).replace([np.inf, -np.inf], np.nan)
    mu = work[ret_col].rolling(50).mean()
    sigma = work[ret_col].rolling(50).std()
    work["ret_zscore"] = (work[ret_col] - mu) / (sigma + 1e-6)
    return work

scored_df = compute_spike_scores(features_df)
print("[SpikeScore] Example tail:")
scored_df[["timestamp", "raw_ret", "ewma_vol", "spike_score", "ret_zscore"]].tail(5)

## 8. Adaptive Thresholding with Dynamic Baseline
Median + MAD adaptive threshold for spike_score to flag candidate bars.

In [ ]:
def mark_spikes(df: pd.DataFrame, score_col: str = "spike_score", window: int = 120, k: float = 3.2) -> pd.DataFrame:
    work = df.copy()
    scores = work[score_col]
    med = scores.rolling(window, min_periods=30).median()
    mad = (scores - med).abs().rolling(window, min_periods=30).median()
    thr = med + k * (mad + 1e-6)
    work["adaptive_thr"] = thr
    work["spike_candidate"] = (scores >= thr).astype(int)
    return work

threshold_df = mark_spikes(scored_df)
print("[Threshold] Candidates count:", int(threshold_df["spike_candidate"].sum()))
threshold_df[["timestamp", "spike_score", "adaptive_thr", "spike_candidate"]].tail(5)

## 9. Label Generation for Supervised Learning
Forward-looking label using future return window and target thresholds.

In [ ]:
def generate_labels(df: pd.DataFrame, close_col: str = "close", future_window: int = 15,
                     target_up: float = 0.01, target_down: float = -0.01) -> pd.DataFrame:
    work = df.copy()
    future_max = work[close_col].rolling(future_window).max().shift(-future_window + 1)
    future_min = work[close_col].rolling(future_window).min().shift(-future_window + 1)
    price_now = work[close_col]
    up_ret = (future_max - price_now) / (price_now + 1e-6)
    down_ret = (future_min - price_now) / (price_now + 1e-6)
    label = ((up_ret >= target_up) | (down_ret <= target_down)).astype(int)
    work["label"] = label
    work["future_max_ret"] = up_ret
    work["future_min_ret"] = down_ret
    return work

labeled_df = generate_labels(threshold_df)
print("[Labels] Positive count:", int(labeled_df["label"].sum()))
labeled_df[["timestamp", "label", "future_max_ret", "future_min_ret"]].head()

## 10. Training Dataset Assembly (Sliding Window)
Construct feature matrix from past N bars aggregated stats; align with labels and split.

In [ ]:
from sklearn.model_selection import train_test_split

AGG_FEATURES = ["raw_ret", "ewma_vol", "spike_score", "ret_zscore", "volume_ratio_20", "spread_pct"]

def build_dataset(df: pd.DataFrame, lookback: int = 10) -> tuple:
    rows = []
    for i in range(lookback, len(df)):
        window = df.iloc[i - lookback : i]
        feats = {}
        for f in AGG_FEATURES:
            if f in window.columns:
                feats[f + "_mean"] = window[f].mean()
                feats[f + "_last"] = window[f].iloc[-1]
        feats["label"] = int(df.iloc[i]["label"]) if "label" in df.columns else 0
        rows.append(feats)
    Xy = pd.DataFrame(rows)
    X = Xy.drop("label", axis=1)
    y = Xy["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = build_dataset(labeled_df)
print("[Dataset]", X_train.shape, X_test.shape, "Pos train:", int(y_train.sum()))
X_train.head(2)